# Lab 06: Bias-Variance Trade-Off Simulation
> Week 6 | CLO2 | ISLP Ch.2.2

**LLo**: คำนวณ MSE, อธิบาย Bias-Variance Trade-Off และประเมินความแม่นยำของ model ได้

---

สัปดาห์นี้เราจะสำรวจหนึ่งในแนวคิดที่สำคัญที่สุดใน Machine Learning คือ **Bias-Variance Trade-Off** แนวคิดหลักคือ error ของ model ประกอบด้วย 3 ส่วน ได้แก่ Bias² (ความเอนเอียงจากการสมมติที่ผิด), Variance (ความไม่เสถียรเมื่อ training data เปลี่ยน) และ Irreducible Error (noise ที่หลีกเลี่ยงไม่ได้) เป้าหมายของ lab นี้คือให้นักศึกษาเห็นด้วยตัวเองผ่าน simulation ว่า U-curve ของ Test MSE เกิดขึ้นได้อย่างไร โดยใช้ polynomial regression degree ต่างๆ และ KNN classification ที่ K ต่างๆ ทักษะนี้จะช่วยให้นักศึกษาตัดสินใจได้ว่า model ซับซ้อนเพียงใดจึงจะเหมาะสมกับข้อมูลที่มี ซึ่งเป็นทักษะพื้นฐานที่ Data Scientist ทุกคนต้องมี

**สิ่งที่จะเรียนรู้:**
- คำนวณ Training MSE และ Test MSE จาก polynomial regression
- Simulate การ decompose error เป็น Bias² + Variance + Irreducible Error
- เห็น U-curve ของ Test MSE เป็น function ของ model complexity
- KNN classification และผลของ K ต่อ bias-variance trade-off

| TODO | ระดับ | หัวข้อ |
|------|-------|--------|
| TODO 1 | ง่าย | คำนวณ Baseline MSE ของ True f |
| TODO 2 | ปานกลาง | เปรียบเทียบ Train/Test MSE ของ Polynomial degrees ต่างๆ |
| TODO 3 | ยาก | วิเคราะห์ผล Bias-Variance Decomposition |
| TODO 4 | ปานกลาง | หา Optimal K สำหรับ KNN classification |
| TODO 5 | ปานกลาง | Case Study: วิเคราะห์ Overfitting ใน Medical Diagnosis |

In [ ]:
# ─── Import libraries ──────────────────────────────────────────────
# วัตถุประสงค์: โหลด libraries ทั้งหมดที่ต้องใช้ใน lab นี้
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.datasets import make_classification

# ─── ตั้งค่า random seed ────────────────────────────────────────────
# วัตถุประสงค์: ให้ผลลัพธ์ reproducible — ทุกคนได้ตัวเลขเดียวกัน
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded successfully.')

## Part 1: สร้าง Synthetic Data จาก f(X) + ε

ในส่วนนี้เราจะสร้างข้อมูลจาก **true function f(X) = sin(X)** โดยเพิ่ม noise ε เข้าไป เพื่อจำลองสถานการณ์จริงที่ข้อมูลมี noise ตามธรรมชาติ เป้าหมายคือให้เห็นว่า irreducible error ที่เกิดจาก Var(ε) เป็นขีดจำกัดที่หลีกเลี่ยงไม่ได้ แม้ใช้ f จริงก็ยังมี error อยู่ ส่วนที่เราปรับปรุงได้คือ reducible error ซึ่งเกิดจาก f̂ ≠ f

**โครงสร้างข้อมูล:**
- `f_true(X) = sin(X)` — true function ที่เราต้องการประมาณ
- ε ~ N(0, 0.5²) — noise, Var(ε) = 0.25
- Y = f_true(X) + ε — observed data
- n = 100 observations, X in [0, 2π]
- Split: 60% train, 40% test

In [ ]:
# ─── กำหนด true function ────────────────────────────────────────────
# วัตถุประสงค์: f_true คือ 'ความจริง' ที่เราต้องการ estimate
def f_true(X):
    return np.sin(X)

# ─── สร้าง observations Y = f_true(X) + noise ──────────────────────
# วัตถุประสงค์: simulate ข้อมูลจริงที่มี noise ซ่อนอยู่
n = 100
noise_std = 0.5
X_all = np.sort(np.random.uniform(0, 2 * np.pi, n))
epsilon = np.random.normal(0, noise_std, n)
y_all = f_true(X_all) + epsilon

# ─── Train/Test Split ──────────────────────────────────────────────
# วัตถุประสงค์: ประเมิน model บน test set ที่ไม่เคยเห็น (ไม่ใช่ training set)
X_train, X_test, y_train, y_test = train_test_split(
    X_all.reshape(-1, 1), y_all, test_size=0.4, random_state=42
)
X_plot = np.linspace(0, 2 * np.pi, 300).reshape(-1, 1)

# ─── Visualize data ────────────────────────────────────────────────
# วัตถุประสงค์: เข้าใจภาพรวมของ dataset ก่อนเริ่ม model fitting
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(X_train, y_train, alpha=0.7, label=f'Train ({len(y_train)} pts)', color='steelblue', s=40)
ax.scatter(X_test,  y_test,  alpha=0.7, label=f'Test ({len(y_test)} pts)', color='coral', s=40, marker='^')
ax.plot(X_plot, f_true(X_plot), 'k-', linewidth=2.5, label='True f(X) = sin(X)', zorder=5)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title('Synthetic Data: Y = sin(X) + e,  e ~ N(0, 0.25)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Training: {len(y_train)} obs  |  Test: {len(y_test)} obs')
print(f'Irreducible Error Var(e) = {noise_std**2:.4f}')
print('→ แม้ใช้ True f ก็ยัง predict ผิดได้ประมาณ 0.25 หน่วย² เพราะ noise')

In [ ]:
# ─── TODO 1 (ง่าย): คำนวณ Baseline MSE ของ True f ─────────────────
# วัตถุประสงค์: หา lower bound ของ Test MSE — นี่คือ irreducible error
# ถ้าเราใช้ f_true เพื่อ predict ผลลัพธ์ควรจะ ≈ Var(ε) = 0.25
#
# ขั้นตอน:
#   1. y_train_pred_true = f_true(X_train).ravel()
#   2. y_test_pred_true  = f_true(X_test).ravel()
#   3. train_mse_true    = mean_squared_error(y_train, y_train_pred_true)
#   4. test_mse_true     = mean_squared_error(y_test,  y_test_pred_true)

# เติม code ของคุณที่นี่:
# y_train_pred_true = ???
# y_test_pred_true  = ???
# train_mse_true    = ???
# test_mse_true     = ???

# uncomment เมื่อเสร็จ:
# print(f'True f Train MSE: {train_mse_true:.4f}')
# print(f'True f Test  MSE: {test_mse_true:.4f}')
# print(f'Theoretical Var(e) = {noise_std**2:.4f}')
# print()
# print('สังเกต: Test MSE ≈ Var(e) ← นี่คือ irreducible error ขั้นต่ำสุด')

## Part 2: Polynomial Regression — Underfitting vs Overfitting

ในส่วนนี้เราจะ fit polynomial model ที่ความซับซ้อน (degree) ต่างกัน 3 ระดับ:
- **Degree 1** (linear): model เรียบง่ายที่สุด → อาจ underfit (high bias)
- **Degree 5** (moderate): flexibility ปานกลาง → น่าจะใกล้ optimal
- **Degree 15** (complex): flexibility สูงมาก → อาจ overfit (high variance)

เป้าหมายคือเห็นว่า Training MSE ลดลงเรื่อยๆ เมื่อ degree เพิ่มขึ้น แต่ Test MSE มี U-shape ซึ่งบ่งบอกว่ามี optimal degree ที่ minimize test error นี่คือ Bias-Variance Trade-Off

**วิธีการ:** ใช้ `make_pipeline(PolynomialFeatures(d), LinearRegression())` ซึ่งรวมขั้นตอน feature expansion และ linear regression เข้าด้วยกันอัตโนมัติ

In [ ]:
# ─── Fit polynomial regression: degree 1, 5, 15 ────────────────────
# วัตถุประสงค์: แสดง visual ของ underfitting, good fit, overfitting
degrees = [1, 5, 15]
colors  = ['royalblue', 'green', 'red']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, (deg, color) in enumerate(zip(degrees, colors)):
    # ─── สร้าง pipeline: PolynomialFeatures → LinearRegression ───────
    # วัตถุประสงค์: make_pipeline ทำ feature transform และ fit ให้อัตโนมัติ
    model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    model.fit(X_train, y_train)

    y_tr_pred = model.predict(X_train)
    y_te_pred = model.predict(X_test)
    tr_mse    = mean_squared_error(y_train, y_tr_pred)
    te_mse    = mean_squared_error(y_test,  y_te_pred)

    # ─── Plot ──────────────────────────────────────────────────────
    axes[i].scatter(X_train, y_train, alpha=0.5, color='steelblue', s=25, label='Train')
    axes[i].scatter(X_test,  y_test,  alpha=0.5, color='coral', s=25, marker='^', label='Test')
    axes[i].plot(X_plot, f_true(X_plot), 'k--', lw=1.5, alpha=0.7, label='True f')
    axes[i].plot(X_plot, model.predict(X_plot), color=color, lw=2.5, label=f'Deg {deg}')
    axes[i].set_title(f'Degree {deg}\nTrain MSE={tr_mse:.3f}  Test MSE={te_mse:.3f}')
    axes[i].set_ylim(-3.5, 3.5)
    axes[i].legend(fontsize=8)
    axes[i].set_xlabel('X')

plt.suptitle('Polynomial Regression: Underfitting vs Optimal vs Overfitting', fontsize=13)
plt.tight_layout()
plt.show()

print('สังเกต: Degree 1 → model ไม่ capture curve ของ sin(X) ← underfit')
print('สังเกต: Degree 15 → model เป็น wiggly มาก ← overfit')

In [ ]:
# ─── TODO 2 (ปานกลาง): ตาราง Train/Test MSE สำหรับ degrees 1-15 ────
# วัตถุประสงค์: เห็น U-curve ของ Test MSE อย่างชัดเจนในรูปแบบตาราง
# ผลลัพธ์ที่คาดหวัง: Train MSE ลดเรื่อยๆ แต่ Test MSE มี minimum ที่ degree ≈ 4-6
#
# ขั้นตอน:
#   1. Loop over degrees_range = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 15]
#   2. สำหรับแต่ละ degree: fit model, คำนวณ train_mse และ test_mse
#   3. เก็บผลใน lists แล้ว print เป็นตาราง

# degrees_range = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 15]
# train_mses, test_mses = [], []
#
# for deg in degrees_range:
#     model = ???
#     model.???
#     tr = ???
#     te = ???
#     train_mses.append(tr)
#     test_mses.append(te)
#
# print(f'{"Degree":>8} | {"Train MSE":>10} | {"Test MSE":>10}')
# print('-' * 35)
# for d, tr, te in zip(degrees_range, train_mses, test_mses):
#     print(f'{d:>8} | {tr:>10.4f} | {te:>10.4f}')

# ─── Plot U-curve ─────────────────────────────────────────────────
# uncomment เมื่อ loop เสร็จแล้ว:
# plt.figure(figsize=(8, 5))
# plt.plot(degrees_range, train_mses, 'b-o', label='Train MSE')
# plt.plot(degrees_range, test_mses,  'r-o', label='Test MSE')
# plt.axhline(y=noise_std**2, color='gray', linestyle='--', label='Irreducible Error')
# plt.xlabel('Polynomial Degree (Flexibility →)')
# plt.ylabel('MSE')
# plt.title('U-curve: Test MSE vs Model Flexibility')
# plt.legend()
# plt.show()

## Part 3: Bias-Variance Decomposition — Simulation จาก 200 Training Sets

ใน Part 2 เราเห็น U-curve จากชุดข้อมูล 1 ชุด แต่ Bias-Variance Decomposition ต้องการการ simulate หลายๆ training sets ถึงจะคำนวณ Bias² และ Variance ได้โดยตรง

**สูตรสำคัญ:**
```
E[(Y - f̂(x₀))²] = Bias²(f̂(x₀)) + Var(f̂(x₀)) + Var(ε)
                    ─────────────   ─────────────   ──────────
                    (mean pred - true f)²   spread ของ preds   noise
```

**วิธี simulate:**
1. สร้าง 200 training sets จาก same true f แต่ noise ต่างกัน
2. สำหรับแต่ละ degree: fit model บนทุก training set → predict บน test points ที่ fixed
3. Bias² = mean((mean_prediction − true_f)²)
4. Variance = mean(variance_of_predictions)
5. Total E[Test MSE] ≈ Bias² + Variance + 0.25

In [ ]:
# ─── Simulation Setup ──────────────────────────────────────────────
# วัตถุประสงค์: กำหนด test points ที่ fixed เพื่อประเมิน E[MSE] อย่าง consistent
X_test_fixed     = np.linspace(0.5, 2*np.pi - 0.5, 50).reshape(-1, 1)
y_test_fixed_true = f_true(X_test_fixed).ravel()

n_sim       = 200
n_train_sim = 60
degrees_sim = [1, 2, 3, 4, 5, 6, 8, 10, 12, 15]
results     = {}

# ─── Simulation Loop ───────────────────────────────────────────────
# วัตถุประสงค์: สำหรับแต่ละ degree fit model บน 200 training sets ต่างกัน
for deg in degrees_sim:
    preds = np.zeros((n_sim, len(X_test_fixed)))

    for sim_i in range(n_sim):
        X_s = np.sort(np.random.uniform(0, 2*np.pi, n_train_sim)).reshape(-1, 1)
        y_s = f_true(X_s).ravel() + np.random.normal(0, noise_std, n_train_sim)
        m   = make_pipeline(PolynomialFeatures(deg), LinearRegression())
        m.fit(X_s, y_s)
        preds[sim_i] = m.predict(X_test_fixed)

    # ─── คำนวณ Bias², Variance, Irreducible ──────────────────────
    # วัตถุประสงค์: decompose E[Test MSE] ออกเป็น 3 ส่วน
    mean_pred = preds.mean(axis=0)
    bias_sq   = np.mean((mean_pred - y_test_fixed_true) ** 2)
    variance  = np.mean(preds.var(axis=0))
    irred     = noise_std ** 2
    results[deg] = {'bias_sq': bias_sq, 'variance': variance,
                    'irred': irred, 'total': bias_sq + variance + irred}

# ─── Plot Bias-Variance Decomposition ─────────────────────────────
# วัตถุประสงค์: visualize ว่าแต่ละ component เปลี่ยนอย่างไรตาม degree
bias_vals  = [results[d]['bias_sq']  for d in degrees_sim]
var_vals   = [results[d]['variance'] for d in degrees_sim]
total_vals = [results[d]['total']    for d in degrees_sim]

plt.figure(figsize=(10, 6))
plt.plot(degrees_sim, bias_vals,  'b-o', label='Bias²', lw=2)
plt.plot(degrees_sim, var_vals,   'r-o', label='Variance', lw=2)
plt.axhline(y=noise_std**2, color='gray', linestyle='--', lw=1.5, label='Irreducible Error (0.25)')
plt.plot(degrees_sim, total_vals, 'k-o', label='Total E[Test MSE]', lw=2.5)
plt.xlabel('Polynomial Degree')
plt.ylabel('Expected MSE')
plt.title('Bias-Variance Decomposition (200 simulations)')
plt.legend()
plt.tight_layout()
plt.show()

print('Degree | Bias²   | Variance | Irred  | Total E[MSE]')
print('-' * 52)
for d in degrees_sim:
    r = results[d]
    print(f'  {d:>3}  | {r["bias_sq"]:>7.4f} | {r["variance"]:>8.4f} | {r["irred"]:.4f} | {r["total"]:>12.4f}')

In [ ]:
# ─── TODO 3 (ยาก): วิเคราะห์ผล Bias-Variance Decomposition ─────────
# วัตถุประสงค์: extract insights จาก simulation results
# ผลลัพธ์ที่คาดหวัง: ระบุ optimal degree, ยืนยัน decomposition, อธิบาย trade-off
#
# ส่วน A — หา optimal degree:
#   optimal_degree = degree ที่ให้ total E[Test MSE] ต่ำสุดใน results
#   hint: ใช้ min(results, key=lambda d: results[d]['total'])
#
# optimal_degree = ???
# min_mse        = ???
# print(f'Optimal degree: {optimal_degree}  |  Min E[Test MSE]: {min_mse:.4f}')
#
# ส่วน B — ยืนยัน decomposition:
#   สำหรับ degree = 1, optimal_degree, 15
#   ตรวจสอบว่า Bias² + Variance + Irred ≈ Total
#
# for d in [1, optimal_degree, 15]:
#     r = results[d]
#     check = r['bias_sq'] + r['variance'] + r['irred']
#     print(f'Degree {d:>2}: Bias²+Var+Irred = {check:.4f}  |  Total = {r["total"]:.4f}')
#
# ส่วน C — ตอบคำถาม:
# ทำไม degree 1 มี Bias² สูง?
# → (เขียนคำตอบใน comment)
# ทำไม degree 15 มี Variance สูง?
# → (เขียนคำตอบใน comment)

## Part 4: KNN Classification — ผลของ K ต่อ Bias-Variance Trade-Off

แนวคิด Bias-Variance Trade-Off ไม่ได้มีแค่ใน Regression แต่ยังใช้ได้กับ Classification ด้วย ใน KNN Classification:
- **K เล็ก** (K=1): decision boundary ซับซ้อน → Low Bias, High Variance → overfit
- **K ใหญ่** (K=100): decision boundary เรียบมาก → High Bias, Low Variance → underfit
- **K เหมาะสม**: balance ระหว่าง Bias และ Variance → minimize Test Error Rate

ใน Part นี้เราจะ:
1. สร้าง synthetic 2-class dataset
2. Visualize decision boundary สำหรับ K=1 และ K=20
3. TODO 4: Plot Train/Test error rate สำหรับ K ตั้งแต่ 1 ถึง 100

In [ ]:
# ─── สร้าง 2-class synthetic dataset ──────────────────────────────
# วัตถุประสงค์: ปัญหา classification ที่มี nonlinear boundary
X_cls, y_cls = make_classification(
    n_samples=300, n_features=2, n_redundant=0, n_informative=2,
    random_state=42, n_clusters_per_class=2, class_sep=0.8
)
X_cls_tr, X_cls_te, y_cls_tr, y_cls_te = train_test_split(
    X_cls, y_cls, test_size=0.35, random_state=42
)

# ─── Plot decision boundaries: K=1 vs K=20 ────────────────────────
# วัตถุประสงค์: เห็นภาพว่า K เล็กทำให้ boundary ซับซ้อน K ใหญ่ทำให้เรียบ
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
x0_min, x0_max = X_cls[:, 0].min() - 0.5, X_cls[:, 0].max() + 0.5
x1_min, x1_max = X_cls[:, 1].min() - 0.5, X_cls[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x0_min, x0_max, 200),
                     np.linspace(x1_min, x1_max, 200))

for i, k in enumerate([1, 20]):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_cls_tr, y_cls_tr)
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    tr_err = 1 - accuracy_score(y_cls_tr, knn.predict(X_cls_tr))
    te_err = 1 - accuracy_score(y_cls_te, knn.predict(X_cls_te))

    axes[i].contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    axes[i].scatter(X_cls_tr[:, 0], X_cls_tr[:, 1], c=y_cls_tr, cmap='RdBu', s=20, alpha=0.7)
    axes[i].set_title(f'K={k}\nTrain Err={tr_err:.3f}  Test Err={te_err:.3f}')
    axes[i].set_xlabel('X1')
    axes[i].set_ylabel('X2')

plt.suptitle('KNN Decision Boundaries: K=1 (Overfit) vs K=20', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ─── TODO 4 (ปานกลาง): Plot Error Rate vs K เพื่อหา Optimal K ────
# วัตถุประสงค์: เห็น U-curve ของ classification error เทียบกับ K
# ผลลัพธ์ที่คาดหวัง: Train Error สูงขึ้นเมื่อ K เพิ่ม, Test Error มี U-shape
#
# K_values = [1, 2, 3, 5, 7, 10, 15, 20, 30, 50, 75, 100]
# train_errors, test_errors = [], []
#
# สำหรับแต่ละ K:
#   1. knn = KNeighborsClassifier(n_neighbors=K)
#   2. knn.fit(X_cls_tr, y_cls_tr)
#   3. train_err = 1 - accuracy_score(y_cls_tr, knn.predict(X_cls_tr))
#   4. test_err  = 1 - accuracy_score(y_cls_te, knn.predict(X_cls_te))

# เติม code ของคุณที่นี่:
# K_values = [1, 2, 3, 5, 7, 10, 15, 20, 30, 50, 75, 100]
# train_errors, test_errors = [], []
# for K in K_values:
#     ???

# Plot:
# plt.figure(figsize=(9, 5))
# plt.plot(K_values, train_errors, 'b-o', label='Train Error Rate')
# plt.plot(K_values, test_errors,  'r-o', label='Test Error Rate')
# plt.xlabel('K (number of neighbors)')
# plt.ylabel('Error Rate')
# plt.title('KNN: Train/Test Error Rate vs K (Flexibility decreases as K increases)')
# plt.legend()
# plt.show()
#
# optimal_K    = K_values[???]  # K ที่ให้ min test error
# min_test_err = ???
# print(f'Optimal K: {optimal_K}  |  Min Test Error: {min_test_err:.4f}')

## Part 5: Case Study — Overfitting ใน Medical Diagnosis

**Scenario:** โรงพยาบาลแห่งหนึ่งต้องการ predict ว่า tumor เป็น malignant หรือ benign จาก biomarker 1 ตัว มีข้อมูล patient เพียง **n = 30 ราย** งานวิจัยภายในรายงาน accuracy 98% บน training set — แต่เมื่อนำไปใช้กับ patient ใหม่ได้แค่ 65% เกิดอะไรขึ้น?

**สิ่งที่จะทำ:**
1. สร้าง dataset ขนาดเล็ก (n=30) ที่ simulate ข้อมูล clinical จริง
2. แสดงว่า polynomial degree สูงๆ overfit อย่างชัดเจนบน small dataset
3. TODO 5: วัด Train vs Test MSE เพื่อยืนยัน overfitting

**สำคัญ**: ใน medical setting การ overfit อาจทำให้ model predict ผิดบน patient จริง ซึ่งอาจเป็นอันตราย — นี่คือเหตุผลที่ต้องประเมิน Test MSE เสมอ

In [ ]:
# ─── สร้าง Small Medical Dataset (n=30) ───────────────────────────
# วัตถุประสงค์: simulate ข้อมูล patient จริงที่มี biomarker (เช่น PSA level)
np.random.seed(7)
n_patients = 30
biomarker  = np.sort(np.random.uniform(0, 10, n_patients))

# True relationship: probability of malignant increases with biomarker (logistic)
p_mal    = 1 / (1 + np.exp(-(biomarker - 5)))
outcomes = (np.random.uniform(size=n_patients) < p_mal).astype(float)

X_bio      = biomarker.reshape(-1, 1)
X_bio_plot = np.linspace(0, 10, 300).reshape(-1, 1)

# ─── Visualize overfitting: degree 1, 3, 12 ───────────────────────
# วัตถุประสงค์: degree 12 จะ fit ทุกจุดในข้อมูล แต่ curve ผิดปกติมาก
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, deg in enumerate([1, 3, 12]):
    model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    model.fit(X_bio, outcomes)
    tr_mse = mean_squared_error(outcomes, model.predict(X_bio).clip(0, 1))
    curve  = model.predict(X_bio_plot).clip(0, 1)

    axes[i].scatter(biomarker, outcomes, alpha=0.8, s=60, c='steelblue', label='Patient data')
    axes[i].plot(X_bio_plot, curve, 'r-', lw=2.5, label=f'Degree {deg}')
    axes[i].axhline(y=0, color='gray', lw=0.5)
    axes[i].axhline(y=1, color='gray', lw=0.5)
    axes[i].set_xlabel('Biomarker Level')
    axes[i].set_ylabel('P(Malignant)')
    axes[i].set_title(f'Degree {deg}  |  Train MSE = {tr_mse:.4f}')
    axes[i].set_ylim(-0.3, 1.3)
    axes[i].legend(fontsize=8)

plt.suptitle('Medical Case Study: Overfitting ใน Small Dataset (n=30)', fontsize=12)
plt.tight_layout()
plt.show()

print('สังเกต: Degree 12 → Train MSE ต่ำมาก แต่ curve ผิดปกติ!')
print('→ นี่คือ overfitting ที่อันตรายใน clinical setting')

In [ ]:
# ─── TODO 5 (ปานกลาง): วัด Train vs Test MSE บน Medical Dataset ──
# วัตถุประสงค์: แสดงว่า Train MSE ต่ำ ≠ good clinical model
# ผลลัพธ์ที่คาดหวัง: degree 12 มี Train MSE ต่ำที่สุด แต่ Test MSE สูงที่สุด
#
# ขั้นตอน:
#   1. แบ่ง data: X_bio_tr, X_bio_te, y_bio_tr, y_bio_te
#      train_test_split(X_bio, outcomes, test_size=0.3, random_state=1)
#   2. Loop over degrees = [1, 2, 3, 5, 8, 12]
#   3. สำหรับแต่ละ degree คำนวณ train_mse และ test_mse
#   4. Print ตาราง

# เติม code ของคุณที่นี่:
# X_bio_tr, X_bio_te, y_bio_tr, y_bio_te = train_test_split(
#     X_bio, outcomes, test_size=0.3, random_state=1
# )
# degrees_med = [1, 2, 3, 5, 8, 12]
# print(f'{"Degree":>8} | {"Train MSE":>10} | {"Test MSE":>10}')
# print('-' * 35)
# for deg in degrees_med:
#     ???
#
# print()
# print('Degree ที่แนะนำสำหรับ clinical use: ???')
# print('เหตุผล: ???')

## Reflection

ตอบคำถามต่อไปนี้ใน markdown cell นี้ (แก้ไขข้อความ `???` ให้เป็นคำตอบของคุณ):

---

### คำถาม 1
จาก simulation ใน Part 3 ถ้าเพิ่ม `n_train_sim` จาก 60 เป็น 600 observations คาดว่า:
- Bias² จะเปลี่ยนอย่างไร? (เพิ่ม / ลด / เท่าเดิม)
- Variance จะเปลี่ยนอย่างไร?
- รูปร่าง U-curve จะเปลี่ยนอย่างไร?

**คำตอบของคุณ:**
> ???

---

### คำถาม 2
ใน Medical Case Study (Part 5) ทำไม polynomial degree 1 (linear) ถึงดีกว่า degree 12 สำหรับ clinical use แม้ว่า Train MSE จะสูงกว่า? เชื่อมกับแนวคิด Bias-Variance Trade-Off อย่างไร?

**คำตอบของคุณ:**
> ???